In [2]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.2/274.2 kB 2.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Cloning into 'detectron2'...
remote: Enumerating objects: 15502, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (201/201), done.
remote: Total 15502 (delta 54), reused 87 (delta 26), pack-reused 15275
Receiving objects: 100% (15502/15502), 6.39 MiB | 18.03 MiB/s, done.
Resolving deltas: 100% (11165/11165), done.


No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61400 sha256=b4c21cf381d90054dd42beb44214d9df994efc311c905f4cbe6049d8db193d87
  Stored in directory: /root/.cache/pip/wheels/01/c0/af/77c1cf53a1be9e42a52b48e5af2169d40ec2e89f7362489dd0
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144554 sha256=6e23c233517d011af6728c8401e20465d827b95c40cf596734161de1e302438c
  Stored in directory: /root/.cache/pip/wheels/12/93/dd/1f6a1

In [3]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0
torch:  2.1 ; cuda:  cu121
detectron2: 0.6


In [4]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

In [5]:
cap = cv2.VideoCapture()
ret, frame = cap.read()

In [ ]:
cfg = get_cfg()
predictor = DefaultPredictor(cfg)
outputs = predictor(frame)

instances = outputs["instances"]

car_instances = instances[instances.pred_classes == 2]
motocycle_instances = instances[instances.pred_classes == 3]
truck_instances = instances[instances.pred_classes == 7]

num_car = len(car_instances)
num_motocycle = len(motocycle_instances)
num_truck = len(truck_instances)

In [ ]:
# Função para configurar linhas no vídeo
def show_video(video_path):
    """
    Configura linha central em um vídeo.

    Parameters:
    - video_path (str): O caminho do arquivo de vídeo.

    Returns:
    - contagem_veiculos (dict): Dicionário com a contagem de carros, motos e caminhões.
    """
    try:
        cap = cv2.VideoCapture(video_path)

        # Verifica se a abertura foi bem-sucedida
        if not cap.isOpened():
            raise Exception(f"Erro ao abrir o vídeo em {video_path}")

        # Obtém as dimensões do vídeo
        width = int(cap.get(3))
        height = int(cap.get(4))

        # Define as coordenadas da linha central
        linha_central = (width // 2, 0, width // 2, height)

        while True:
            # Lê o próximo frame
            ret, frame = cap.read()

            # Verifica se chegou ao final do vídeo
            if not ret:
                break

            # Desenha a linha central no frame
            cv2.line(frame, (linha_central[0], linha_central[1]), (linha_central[2], linha_central[3]), (255, 0, 255), 2)

            cfg = get_cfg()
            predictor = DefaultPredictor(cfg)
            outputs = predictor(frame)

            instances = outputs["instances"]

            car_instances = instances[instances.pred_classes == 2]
            motocycle_instances = instances[instances.pred_classes == 3]
            truck_instances = instances[instances.pred_classes == 7]

            num_car = len(car_instances)
            num_motocycle = len(motocycle_instances)
            num_truck = len(truck_instances)

            # Verificar se algum veículo cruzou a linha central
            for instance in instances:
                bbox = instance.pred_boxes.tensor.numpy()[0]
                x_center = (bbox[0] + bbox[2]) / 2
                y_center = (bbox[1] + bbox[3]) / 2

                # Verifica se o centro do veículo está na linha central
                if linha_central[0] <= x_center <= linha_central[2] and linha_central[1] <= y_center <= linha_central[3]:
                    if instance.pred_classes == 2:  # Carro
                        contagem_veiculos['car'] += 1
                    elif instance.pred_classes == 3:  # Moto
                        contagem_veiculos['moto'] += 1
                    elif instance.pred_classes == 7:  # Caminhão
                        contagem_veiculos['caminhao'] += 1

            # Mostra o frame com a linha central desenhada
            cv2_imshow(frame)
            if cv2.waitKey(30) & 0xFF == 27:  # Pressione 'Esc' para sair
                break

        cap.release()
        cv2.destroyAllWindows()
        return contagem_veiculos

    except Exception as e:
        print(f"Erro: {e}")
        return None

show_video("/content/drive/MyDrive/Colab Notebooks/rua.mp4")

Erro: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx
